In [43]:
import pandas as pd 
import numpy as np
import re
from pathlib import Path

import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display

import matplotlib.pyplot as plt

In [37]:
sample_catalog_df = pd.read_csv('sample_sweetcat.csv')
corrected_path = Path('observations/Best_RM/linear_corrected')


def normalize_star_name(star_name):
    return re.sub(r'[^a-z0-9]', '', str(star_name).lower())


corrected_files = sorted(corrected_path.glob('*_linear_corrected.csv'))

observations_df = pd.DataFrame(
    [
        {
            'corrected_file': obs_path.name,
            'observation_file': obs_path.name.replace('_linear_corrected.csv', '.rdb'),
            'star_key': normalize_star_name(
                re.sub(r'_esp.*$', '', obs_path.name.replace('_linear_corrected.csv', ''), flags=re.IGNORECASE)
            ),
        }
        for obs_path in corrected_files
    ]
)
sample_catalog_with_keys = sample_catalog_df.assign(
    star_key=sample_catalog_df['star'].map(normalize_star_name)
)

observation_sample_df = observations_df.merge(
    sample_catalog_with_keys,
    on='star_key',
    how='left',
).drop(columns='star_key')
observation_sample_df = observation_sample_df[['corrected_file', 'observation_file', *sample_catalog_df.columns]]

corrected_tables = []
for obs_path in corrected_files:
    obs_df = pd.read_csv(obs_path)
    obs_name = obs_path.name.replace('_linear_corrected.csv', '.rdb')
    obs_df['corrected_file'] = obs_path.name
    obs_df['observation_file'] = obs_name
    obs_df['star_key'] = normalize_star_name(
        re.sub(r'_esp.*$', '', obs_name.replace('.rdb', ''), flags=re.IGNORECASE)
    )
    corrected_tables.append(obs_df)

rm_df = pd.concat(corrected_tables, ignore_index=True)
rm_df = rm_df.merge(sample_catalog_with_keys, on='star_key', how='left').drop(columns='star_key')

print(f'Loaded {len(corrected_files)} corrected observations')
print(f'Total spectra/rows: {len(rm_df)}')
observation_sample_df.head(2)

Loaded 22 corrected observations
Total spectra/rows: 1416


,corrected_file,observation_file,star,Name,gaia_dr3,Teff,eTeff,Logg,eLogg,[Fe/H],...,Gmag,Plx,Distance,Mass_t,Radius_t,SWFlag,Reference,Link,Update,Database
0,HD189733_esp19_3_linear_corrected.csv,HD189733_esp19_3.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA
1,HD189733_esp19_4_linear_corrected.csv,HD189733_esp19_4.rdb,HD189733,HD 189733,1827242816201846144,4969.0,48.0,4.3,0.12,-0.08,...,7.4284,50.5668,19.775821,0.751097,0.786858,1,Sousa et al. 2021,https://ui.adsabs.harvard.edu/abs/2021arXiv210...,2021-01-01,EU;NASA


In [38]:
def plot_corrected_observation(obs_name):
    obs_df = rm_df.loc[rm_df['observation_file'] == obs_name].sort_values('rjd')

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=obs_df['rjd'],
            y=obs_df['true_vrad'],
            mode='markers+lines',
            marker=dict(size=7),
            name='true_vrad',
            hovertemplate='rjd=%{x}<br>true_vrad=%{y}<extra></extra>',
        )
    )
    fig.update_layout(
        title=f'Linear-trend corrected RV: {obs_name}',
        xaxis_title='rjd',
        yaxis_title='true_vrad',
        template='plotly_white',
        hovermode='closest',
    )

    fig.show()


obs_selector = widgets.Dropdown(
    options=sorted(rm_df['observation_file'].unique()),
    description='obs_name:',
)
interactive_plot = widgets.interactive_output(plot_corrected_observation, {'obs_name': obs_selector})
display(obs_selector, interactive_plot)

Dropdown(description='obs_name:', options=('HD189733_esp19_3.rdb', 'HD189733_esp19_4.rdb', 'HD209458_esp19_1.r…

Output()

In [47]:
target_column = 'true_vrad'
feature_columns = ['fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca']

for column in feature_columns + [target_column]:
    rm_df[column] = pd.to_numeric(rm_df[column], errors='coerce')

ml_df = rm_df.dropna(subset=[target_column]).reset_index(drop=True)
X = ml_df[feature_columns]
y = ml_df[target_column]
groups = ml_df['observation_file']

missing_summary = X.isna().sum().rename('n_missing').to_frame()
missing_summary['missing_fraction'] = missing_summary['n_missing'] / len(X)

print(f'ML rows: {len(ml_df)}')
print(f'Features: {feature_columns}')
print(f'Target: {target_column}')
display(missing_summary)
ml_df[['observation_file', 'star', 'rjd', target_column, *feature_columns]].head()

ML rows: 1416
Features: ['fwhm', 'bis_span', 'contrast', 's_mw', 'ha', 'na', 'ca']
Target: true_vrad


,n_missing,missing_fraction
fwhm,0,0.0
bis_span,0,0.0
contrast,0,0.0
s_mw,0,0.0
ha,0,0.0
na,0,0.0
ca,0,0.0


,observation_file,star,rjd,true_vrad,fwhm,bis_span,contrast,s_mw,ha,na,ca
0,HD189733_esp19_3.rdb,HD189733,59437.538315,48.11,7477.005863,-32.279851,57.246647,516.566455,0.349360,0.182110,0.465102
1,HD189733_esp19_3.rdb,HD189733,59437.542268,45.83,7475.083070,-31.580881,57.238499,517.318432,0.350085,0.181946,0.465822
2,HD189733_esp19_3.rdb,HD189733,59437.546398,43.45,7474.116028,-32.931664,57.242054,518.070345,0.350829,0.181935,0.466542
3,HD189733_esp19_3.rdb,HD189733,59437.550744,40.94,7474.717020,-32.177716,57.256935,518.546523,0.350409,0.181665,0.466999
4,HD189733_esp19_3.rdb,HD189733,59437.554865,38.56,7475.447079,-32.909677,57.246594,517.067939,0.348935,0.181384,0.465582


In [40]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

preprocess = ColumnTransformer(
    transformers=[
        ('activity_features', SimpleImputer(strategy='median'), feature_columns),
    ],
    remainder='drop',
)

baseline_model = Pipeline(
    steps=[
        ('preprocess', preprocess),
        ('model', RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=3)),
    ]
)

baseline_model.fit(X.iloc[train_idx], y.iloc[train_idx])
y_pred = baseline_model.predict(X.iloc[test_idx])

metrics = pd.Series(
    {
        'mae': mean_absolute_error(y.iloc[test_idx], y_pred),
        'rmse': mean_squared_error(y.iloc[test_idx], y_pred) ** 0.5,
        'r2': r2_score(y.iloc[test_idx], y_pred),
        'n_train_rows': len(train_idx),
        'n_test_rows': len(test_idx),
        'n_train_observations': groups.iloc[train_idx].nunique(),
        'n_test_observations': groups.iloc[test_idx].nunique(),
    }
)

display(metrics.to_frame('value'))

,value
mae,8.630670
rmse,14.977652
r2,-0.050095
n_train_rows,1064.000000
n_test_rows,352.000000
n_train_observations,16.000000
n_test_observations,6.000000
